# Evaluation on Gemma-3-4b-it using FLORES+ benchmark for translation tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.9 MB/s eta 0:00:00


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


In [3]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00


In [4]:
import torch
from datasets import load_dataset
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
ds_mkd = load_dataset("openlanguagedata/flores_plus", "mkd_Cyrl")

README.md:   0%|          | 0.00/73.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/535k [00:00<?, ?B/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/559k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [8]:
ds_eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn")

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

eng_Latn.jsonl:   0%|          | 0.00/423k [00:00<?, ?B/s]

eng_Latn.jsonl:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [9]:
ds_mkd

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [10]:
ds_eng

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [11]:
dataset_mkd = ds_mkd['dev'].to_pandas()
dataset_mkd

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,mkd,Cyrl,mace1250,,Во понеделникот научниците од медицинскиот фак...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,mkd,Cyrl,mace1250,,Водечките истражувачи сметаат дека ова може да...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,mkd,Cyrl,mace1250,,JAS 39C Грипен се сруши на писта околу 9:30 am...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,mkd,Cyrl,mace1250,,"Утврдено е дека пилот бил Дилокрит Патави, вод...",https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,mkd,Cyrl,mace1250,,Локалните медиуми известуваат за аеродромско п...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,mkd,Cyrl,mace1250,,Туристичката сезона на ридските станици во гла...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,mkd,Cyrl,mace1250,,"Меѓутоа, имаат поинаква убавина и шарм во зима...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,mkd,Cyrl,mace1250,,Само неколку авиокомпании сѐ уште нудат тарифи...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,mkd,Cyrl,mace1250,,Меѓу авиокомпаниите што го нудат тоа се „Ер Ка...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [12]:
dataset_eng = ds_eng['dev'].to_pandas()
dataset_eng

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,eng,Latn,stan1293,,"On Monday, scientists from the Stanford Univer...",https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,eng,Latn,stan1293,,Lead researchers say this may bring early dete...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,eng,Latn,stan1293,,The JAS 39C Gripen crashed onto a runway at ar...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,eng,Latn,stan1293,,The pilot was identified as Squadron Leader Di...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,eng,Latn,stan1293,,Local media reports an airport fire vehicle ro...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,eng,Latn,stan1293,,The tourist season for the hill stations gener...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,eng,Latn,stan1293,,"However, they have a different kind of beauty ...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,eng,Latn,stan1293,,Only a few airlines still offer bereavement fa...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,eng,Latn,stan1293,,"Airlines that offer these include Air Canada, ...",https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [13]:
text_mkd = dataset_mkd['text'].values.tolist()[:100]
text_eng = dataset_eng['text'].values.tolist()[:100]

In [14]:
len(text_mkd)

100

In [15]:
len(text_eng)

100

In [16]:
text_mkd[:10]

['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.',
 'Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.',
 'JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.',
 'Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.',
 'Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.',
 '28-годишниот Вида

In [17]:
text_eng[:10]

['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.',
 'Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'The pilot was identified as Squadron Leader Dilokrit Pattavee.',
 'Local media reports an airport fire vehicle rolled over while responding.',
 '28-year-old Vidal had joined Barça three seasons ago, from Sevilla.',
 'Since moving to the Catalan-capital, Vidal had played 49 games for the club.',
 "The protest started around 11:00 local time (UTC+1) on 

In [18]:
model_name = "google/gemma-3-4b-it"

In [19]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [21]:
def get_prompt_mkd(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"


In [22]:
def get_prompt_srb(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"

In [23]:
def get_prompt_bug(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОтговор ({language}):"

In [24]:
def get_prompt_eng(instruction,text,language):
  return f"{instruction}\nText:{text}\nResponse ({language}):"

In [26]:
def batch_translate(texts,instruction, batch_size=8, max_new_tokens=256,lang_code="eng",language="English"):
    all_preds = []
    tokenizer.pad_token = tokenizer.eos_token
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        if lang_code=="mkd":
          prompts = [[{"role": "user", "content":get_prompt_mkd(instruction,t,language)}] for t in batch]
        elif lang_code == "bug":
          prompts = [[{"role": "user", "content":get_prompt_bug(instruction,t,language)}] for t in batch]
        elif lang_code == "srb":
          prompts = [[{"role": "user", "content":get_prompt_srb(instruction,t,language)}] for t in batch]
        else:
          prompts = [[{"role": "user", "content":get_prompt_eng(instruction,t,language)}] for t in batch]

        inputs = tokenizer.apply_chat_template(
            prompts,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        input_len = inputs.input_ids.shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

        for translation in decoded:
            print(f"Batch {i}")
            clean_translation = translation.split("\n")[0].strip()
            all_preds.append(clean_translation)
            print(f"Result: {clean_translation}\n")

    return all_preds



In [27]:
from nltk.tokenize import sent_tokenize

def first_sentence(text):
    sents = sent_tokenize(text)
    return sents[0] if sents else text

In [28]:
bleu = load('bleu')

## Few-Shot Mono-lingual prompt (MKD->ENG)

In [29]:
instruction = f"""Преведи го следниот текст од **Македонски** во **Англиски**.Само испечати го преводот на англиски. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Одговор (Англиски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Пример 2
  Македонски: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Одговор (Англиски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Пример 3
  Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Одговор (Англиски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Пример 4
  Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Одговор (Англиски): This study focuses on the development of new methods for data analysis.
  Пример 5
  Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Одговор (Англиски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [30]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="mkd",language="Англиски")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 0
Result: On Monday, scientists from Stanford University’s medical school announced that they had invented a new tool for diagnosing purposes, allowing cells to be sorted by type. It is a small chip for printing that can be produced using standard ink-jet printers, and costs approximately one US cent.

Batch 0
Result: Leading researchers believe this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in low-income countries, where survival rates from diseases such as breast cancer may be twice as low as in wealthier nations.

Batch 0
Result: The JAS 39 Gripen crashed on the runway around 9:30 AM local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.

Batch 0
Result: It was ascertained that the pilot was Dilokrit Patavi, the squadron leader.

Batch 0
Result: Local media report on an airport fire truck that overturned while firefighters responded to the call.

Batch 0
Result: 28-year-old Vidal from

In [31]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [34]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.2996014953454786,
 'precisions': [0.6294277929155313,
  0.3689752936411503,
  0.23427606585056987,
  0.1480828558836492],
 'brevity_penalty': 1.0,
 'length_ratio': 1.025958466453674,
 'translation_length': 2569,
 'reference_length': 2504}

In [35]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 54.08


## Few-Shot Cross-lingual prompt (MKD->ENG) in english language

In [36]:
instruction = f"""Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Response (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Example 2
  Macedonian: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Response (English): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Example 3
  Macedonian: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Response (English): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Example 4
  Macedonian: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Response (English): This study focuses on the development of new methods for data analysis.
  Example 5
  Macedonian: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Response (English): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [37]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128)

Batch 0
Result: On Monday, scientists from Stanford University’s medical school announced that they had invented a new tool for diagnosing, whereby cells are sorted according to types: it’s a small chip for printing that can be produced using standard ink-jet printers, and at a price of about one US cent.

Batch 0
Result: Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in low-income countries, where the survival rate of diseases such as breast cancer may be twice as low as in wealthier nations.

Batch 0
Result: A JAS 39C Gripen crashed on the runway around 9:30 AM local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.

Batch 0
Result: It has been determined that the pilot was Dilokrit Patavi, the squadron leader.

Batch 0
Result: Local media report on an airport fire truck that overturned while firefighters responded to the call.

Batch 0
Result: 28-year-old Vid

In [38]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [39]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.3158559721082145,
 'precisions': [0.6361185983827493,
  0.37685222266720064,
  0.2503128911138924,
  0.16586852416195036],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0371405750798721,
 'translation_length': 2597,
 'reference_length': 2504}

In [40]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 55.34


## Few-Shot Cross-lingual prompt (MKD->ENG) in serbian language

In [41]:
instruction= f"""Преведи следећи текст са **Македонског** на **Енглески** језик.
Само испиши превод. Не додавај никаква објашњења.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Одговор (Енглески): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Одговор (Енглески): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Одговор (Енглески): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Одговор (Енглески): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""


In [42]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="srb",language="Енглески")

Batch 0
Result: On Monday, scientists from Stanford University’s medical school announced they had invented a new tool for diagnosing diseases by sorting cells according to their types. It’s a small chip for printing that can be produced using standard ink-jet printers, costing around one US cent.

Batch 0
Result: Leading researchers believe this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in low-income countries, where survival rates from diseases such as breast cancer may be twice as low as in wealthier nations.

Batch 0
Result: A JAS 39C Gripen crashed around 9:30 AM local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.

Batch 0
Result: It has been determined that the pilot was Dilokrit Patavi, the squadron leader.

Batch 0
Result: Local media report on an airport fire truck that overturned while firefighters responded to the call.

Batch 0
Result: 28-year-old Vidal from Seville joined Barce

In [43]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [44]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.30872810028985576,
 'precisions': [0.6340607950116913,
  0.3767234387672344,
  0.24344885883347422,
  0.1562224183583407],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0247603833865815,
 'translation_length': 2566,
 'reference_length': 2504}

In [45]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 53.90


## Few-Shot Cross-lingual prompt (MKD->ENG) in bulgarian language

In [46]:
instruction = f"""Преведи следния текст от **Македонски** на **Английски** език.
Само отпечатай превода на английски. Не добавяй никакви обяснения.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Отговор (Английски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Отговор (Английски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Отговор (Английски): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Отговор (Английски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [47]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="bug",language="Английски")

Batch 0
Result: On Monday, scientists from Stanford University’s medical school announced that they had invented a new tool for diagnosing purposes, whereby cells are sorted according to types: it is a small chip for printing that can be produced using standard ink-jet printers, and at a price of about one US cent.

Batch 0
Result: Leading researchers believe this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in low-income countries, where survival rates from diseases such as breast cancer may be twice as low as in wealthier nations.

Batch 0
Result: A JAS 39C Gripen crashed near 9:30 AM local time (2:30 UTC) and exploded, resulting in the airport being closed to commercial flights.

Batch 0
Result: It has been established that the pilot was Dilokrit Patavi, the squadron leader.

Batch 0
Result: Local media report on an airport fire truck that overturned while firefighters responded to the call.

Batch 0
Result: 28-year-old Vidal from Sev

In [48]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [49]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.31541998105181024,
 'precisions': [0.6354166666666666,
  0.3812199036918138,
  0.2504180602006689,
  0.1631762652705061],
 'brevity_penalty': 1.0,
 'length_ratio': 1.035143769968051,
 'translation_length': 2592,
 'reference_length': 2504}

In [50]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 55.54


## Few-Shot Mono-lingual prompt (ENG->MKD)

In [51]:
instruction = f"""Translate the following text from **English** into **Macedonian**. Only output the translation. Do not add explanations.
  Example 1
  English: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Response (Macedonian): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Example 2
  English: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Response (Macedonian): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Example 3
  English: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Response (Macedonian): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Example 4
  English: This study focuses on the development of new methods for data analysis.
  Response (Macedonian)): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Example 5
  English: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Response (Macedonian): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [52]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,language="Macedonian")

Batch 0
Result: Во вторник, научници од Школот за медицина при Универзитетот Стенфорд објавија дека произведени е нов дијагностички инструмент кој може да ги сортира клетките според вид: мал, штамлив чип кој може да биде произведен користејќи стандардни струјни принтери за можноста да се направи за околу една американска центар.

Batch 0
Result: Главните истражувачи велат дека ова може да доведе до ран детектирање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со низок доход, каде што стапките на преживување за болести како ракот на матката можат да бидат половина од оние во богатите земји.

Batch 0
Result: Јасе 39Ц Грипен се урна на писта околу 9:30 наутро локално време (0230 UTC) и експлодираше, затворајќи го адиропорт за комерцијални полети.

Batch 0
Result: Пилотот беше идентификуван како Скваерн Лиддер Дилиокрит Паттавеи.

Batch 0
Result: Локалните медиуми ја сообщаваат пожарната возилка на авионската леталиштем наскоро се преклопувала додека реагирала.

Batch 0
Res

In [53]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [54]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.18143717987881253,
 'precisions': [0.5106811769447803,
  0.23981520369592607,
  0.13327487943884261,
  0.07290233837689133],
 'brevity_penalty': 0.9768934704505555,
 'length_ratio': 0.9771563607719574,
 'translation_length': 2481,
 'reference_length': 2539}

In [55]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 46.60


## Few-Shot Cross-lingual prompt (ENG->MKD) in macedonian language

In [56]:
instruction = f"""Преведи го следниот текст од **Англиски** во **Македонски**. Само испечати го преводот на македонски јазик. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Англиски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Одговор (Македонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Пример 2
  Англиски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Одговор (Македонски): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Пример 3
  Англиски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Одговор (Македонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Пример 4
  Англиски: This study focuses on the development of new methods for data analysis.
  Одговор (Македонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Пример 5
  Англиски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Одговор (Македонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [57]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="mkd",language="Македонски")

Batch 0
Result: Во вторник, научници од Школот за медицина на Универзитетот Стенфорд објавија за откритие на нова дијагностичка алатка која може да ги сортира клетките по вид: мал принтер кој може да се произведе користејќи стандардни струјни принтери за можноста да се направи за околу една американска цента по парче.

Batch 0
Result: Главните истражувачи велат дека ова може да доведе до ранко откривање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со низок доход, каде што стапките на преживување за болести како ракот на матката честопати бидат половина од оние во богатите земји.

Batch 0
Result: Јасе 39Ц Грипен се урна на писта околу 9:30 часот локално време (0230 UTC) и експлодираше, што го затвори аеролифтот за комерцијални полети.

Batch 0
Result: Пилото беше идентификуван како Сквадрон лидер Дилиокрит Паттавеи.

Batch 0
Result: Локалните медиуми ја сообщаваат пожарната возилка на авионската леталиштем набавка се преклошила додека реагираше.

Batch 0
Result: Видал, 

In [58]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [59]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.18670442399530277,
 'precisions': [0.5127996749288907,
  0.24904701397712833,
  0.13976116762494473,
  0.07727903748264692],
 'brevity_penalty': 0.968802570797529,
 'length_ratio': 0.9692792437967703,
 'translation_length': 2461,
 'reference_length': 2539}

In [60]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 47.32


## Few-Shot Cross-lingual prompt (ENG->MKD) in serbian language

In [61]:
instruction = f"""Преведи следећи текст са **Eнглеског** на **Mакедонски** језик.
Само испиши превод на македонски језик. Не додавај никаква објашњења или додатне примере.

Пример 1
Енглески: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Одговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Енглески: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Одговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Енглески: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Одговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Енглески: This study focuses on the development of new methods for data analysis.
Одговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Енглески: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Одговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [62]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="srb",language="Македонски")

Batch 0
Result: На понеделник, научници од Школот за медицина при Универзитетот Стенфорд ја анонсираа иновацијата на нова дијагностичка алатка која може да сортира клетки според вид: мал принтер кој може да се произведе користејќи стандардни струјни принтери за можноста да се направи за околу една американска центар.

Batch 0
Result: Главните истражувачи велат дека ова може да доведе до ран детектирање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со низок доход, каде што стопата на преживување за болести како ракот на матката може да биде половина од онаа во богатите земји.

Batch 0
Result: JAS 39C Gripen се урна на писта околу 9:30 бирол локално време (0230 UTC) и експлодираше, што го затвори авиолетот за комерцијални летови.

Batch 0
Result: Пилото беше идентификуван како Сквадрон лидер Дилиокрит Паттавеи.

Batch 0
Result: Локалните медиуми ја сообщаваат пожарната возилка на авионската леталиштем располагање се преклопувала додека реагирала.

Batch 0
Result: Видал, к

In [63]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [64]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.1764252625298047,
 'precisions': [0.5010150223304912,
  0.23487092678798138,
  0.1316836058329651,
  0.07073509015256588],
 'brevity_penalty': 0.9696145293904797,
 'length_ratio': 0.9700669554942891,
 'translation_length': 2463,
 'reference_length': 2539}

In [65]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 45.90


## Few-Shot Cross-lingual prompt (ENG->MKD) in bulgarian language

In [66]:
instruction = f"""Преведи следния текст от **Aнглийски** на **Mакедонски** език.
Само отпечатай превода на македонски език. Не добавяй никакви обяснения или допълнителни примери.

Пример 1
Английски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Отговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Английски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Отговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Английски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Отговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Английски: This study focuses on the development of new methods for data analysis.
Отговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Английски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Отговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""


In [67]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="bug",language="Македонски")

Batch 0
Result: Во вторник, научници од Школот за медицина на Универзитетот Стенфорд го анонсираа иновацијата на нова дијагностичка алатка која може да ги сортира клетките според вид: мал принтер кој може да се произведе користејќи стандардни струјни принтери за можноста да се направи за околу една американска центар.

Batch 0
Result: Главните истражувачи велат дека ова може да доведе до ранко детектирање на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со низок доход, каде што стапките на преживување за болести како ракот на матката честопати бидат половина од оние во богатите земји.

Batch 0
Result: Јасе 39Ц Грипен се урна на писта околу 9:30 биро локално време (0230 UTC) и експлодираше, затворајќи ги авиолетите.

Batch 0
Result: Пилото беше идентификуван како Сквадрон лидер Дилиокрит Паттавеи.

Batch 0
Result: Локалните медиуми ја сообщаваат пожарната возилка на авионската леталиштем набавка се преклопи додека реагираше.

Batch 0
Result: Видал, кој е на 28 години, се пр

In [68]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [69]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.1794183485813784,
 'precisions': [0.5044462409054163,
  0.2337826453243471,
  0.1310466138962181,
  0.07448275862068965],
 'brevity_penalty': 0.974068896988146,
 'length_ratio': 0.974399369830642,
 'translation_length': 2474,
 'reference_length': 2539}

In [70]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 46.09
